#Bert with Enhanced Typed Entity Markers

#Introduction

The notebook is structured as follows:
  1. Setup
  2. Evaluating the model
  3. Model inference on a sentence

###Colab setup -- clone repo

In [ ]:
GIT_KEY=...#your key here
REPO_URL="github.com/jonathondilworth/uom-relation-extraction.git"

!git clone https://$GIT_KEY@$REPO_URL

Cloning into 'uom-relation-extraction'...
remote: Enumerating objects: 304, done.
remote: Counting objects: 100% (304/304), done.
remote: Compressing objects: 100% (193/193), done.
remote: Total 304 (delta 142), reused 255 (delta 101), pack-reused 0 (from 0)
Receiving objects: 100% (304/304), 9.81 MiB | 17.03 MiB/s, done.
Resolving deltas: 100% (142/142), done.


In [6]:
%cd uom-relation-extraction


/content/uom-relation-extraction/uom-relation-extraction


###Switch to correct branch (delete when everything is on main)

In [3]:
!git branch -r

  origin/HEAD -> origin/main
  origin/am-cgcn-experimentation
  origin/jd-baseline-changes
  origin/jd-checkpoint-load-and-eval
  origin/jd-init
  origin/jd-re-baseline-src
  origin/main
  origin/ra-cgcn


In [4]:
!git checkout -b jd-checkpoint-load-and-eval origin/jd-checkpoint-load-and-eval


Branch 'jd-checkpoint-load-and-eval' set up to track remote branch 'jd-checkpoint-load-and-eval' from 'origin'.
Switched to a new branch 'jd-checkpoint-load-and-eval'


###Dataset setup instructions

Before running the training scripts, make sure your dataset files are placed inside the repository with the following structure:

```
uom-relation-extraction/
│── data/
│   ├── retacred/
│   │   ├── train.json
│   │   ├── dev.json
│   │   ├── test.json
│   ├── tacred/
│   │   ├── train.json
│   │   ├── dev.json
│   │   ├── test.json
│   │   ├── dev_rev.json
│   │   ├── test_rev.json  
```


###Install requirements (TODO: put requirements.txt in the directory)

In [8]:
!pip install -r "requirements.txt" > /dev/null 2>&1

In [35]:
#pick seed
SEED= 78

In [36]:
!python src/train_retacred.py \
    --model_name_or_path bert-base-cased \
    --input_format typed_entity_marker_pos \
    --seed $SEED \
    --data_dir ./data/retacred \
    --train_batch_size 64 \
    --test_batch_size 64 \
    --learning_rate 5e-5 \
    --gradient_accumulation_steps 1 \
    --run_name bert-base


/content/uom-relation-extraction/uom-relation-extraction/src/model.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast()
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: 3
wandb: You chose "Don't visualize my results"
wandb: Tracking run with wandb version 0.19.7
wandb: W&B syncing is set to `offline` in this directory.  
wandb: Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
2025-03-06 14:52:37.754315: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-06 14:52:37

###Evaluation

In [37]:
!python src/inference.py --data_dir data/retacred --model_checkpoint saved_models/checkpoint-4000.pt --load_path saved_models

/content/uom-relation-extraction/uom-relation-extraction/src/model.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast()
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_http.py", line 406, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.11/dist-packages/requests/models.py", line 1024, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/saved_models/resolve/main/config.json

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/transformers/utils/hub.py", line 403, in cached_file
    resolved_file = hf_hub_download(
                    ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packa

#Model inference on a sentence

In this section, you can experiment with our model's predictions by inputting your own sentence and related information.  
This allows you to observe how the model processes different syntactic structures and relationships.  

### Example 1:

`Elon Musk is the CEO of Tesla`  

- **Subject:** `Elon Musk`  
- **Subject Type:** `PERSON`  
- **Object:** `Tesla`  
- **Object Type:** `ORGANIZATION`  
- **POS Tags:** `NNP, NNP, VBZ, DT, NN, IN, NNP`  
- **Dependency Relations:** `nsubj, compound, ROOT, det, attr, case, nmod`  

### Example 2:

`Tom Thabane resigned in October last year to form the All Basotho Convention (ABC), crossing the floor with 17 members of parliament, causing constitutional monarch King Letsie III to dissolve parliament and call the snap election.`  

- **Subject:** `All Basotho Convention`  
- **Subject Type:** `ORGANIZATION`  
- **Object:** `Tom Thabane`  
- **Object Type:** `PERSON`  
- **POS Tags:** `NNP, NNP, VBD, IN, NNP, JJ, NN, TO, VB, DT, DT, NNP, NNP, -LRB-, NNP, -RRB-, ,, VBG, DT, NN, IN, CD, NNS, IN, NN, ,, VBG, JJ, NN, NNP, NNP, NNP, TO, VB, NN, CC, VB, DT, NN, NN, .`  
- **Dependency Relations:** `compound, nsubj, ROOT, case, nmod, amod, nmod:tmod, mark, xcomp, det, compound, compound, dobj, punct, appos, punct, punct, xcomp, det, dobj, case, nummod, nmod, case, nmod, punct, xcomp, amod, compound, compound, compound, dobj, mark, xcomp, dobj, cc, conj, det, compound, dobj, punct`  




In [24]:
def find_entity_indices(tokens, entity):
    entity_tokens = entity.split()
    entity_length = len(entity_tokens)

    for i in range(len(tokens) - entity_length + 1):
        if tokens[i : i + entity_length] == entity_tokens:
            #returns start and end index
            return i, i + entity_length - 1
    #return this if the entity is not in sentence
    return -1, -1

In [25]:
import subprocess
import ipywidgets as widgets
from IPython.display import display

def on_submit(b):
    sentence = sentence_box.value.strip()
    subject = subject_box.value.strip()
    subject_type = subject_type_box.value.strip()
    object_ = object_box.value.strip()
    object_type = object_type_box.value.strip()
    pos_tags = pos_box.value.strip()
    deprel_tags = deprel_box.value.strip()

    print("Thinking....")

    if not all([sentence, subject, subject_type, object_, object_type, pos_tags, deprel_tags]):
        print("Please enter all values!")
        return

    #tokenize
    tokens = sentence.split()
    tokens_str = ",".join(tokens)
    pos_list = pos_tags.split(",")
    deprel_list = deprel_tags.split(",")

    #check that pos/deprel/token length is the same
    if len(pos_list) != len(tokens) or len(deprel_list) != len(tokens):
        print("POS tags and dependency relations must match the number of tokens")
        return

    #find indices for subject and object
    subj_start, subj_end = find_entity_indices(tokens, subject)
    obj_start, obj_end = find_entity_indices(tokens, object_)
    #check to make sure word exists
    if subj_start == -1 or obj_start == -1:
        print("Could not find subject or object in the sentence. Please try again.")
        return

    #create command
    command = [
        "python", "src/inference_sent.py",
        "--sentence", sentence,
        "--tokens", tokens_str,
        "--pos", ",".join(pos_list),
        "--deprel", ",".join(deprel_list),
        "--subj_start", str(subj_start),
        "--subj_end", str(subj_end),
        "--obj_start", str(obj_start),
        "--obj_end", str(obj_end),
        "--subj_type", subject_type,
        "--obj_type", object_type
    ]
    try:
        result = subprocess.run(command, capture_output=True, text=True, check=True)

        result_output = widgets.Output(
        layout=widgets.Layout(width='90%', border='1px solid black')
    )
        display(result_output)
        with result_output:
            print("RESULT")
            print(result.stdout)
    except subprocess.CalledProcessError as e:
        print("Error Running Model:")
        print(e.stderr)



    #DEBUGGING
    # print(f"Sentence: {sentence}")
    # print(f"Subject: {subject} ({subject_type}) → Start: {subj_start}, End: {subj_end}")
    # print(f"Object: {object_} ({object_type}) → Start: {obj_start}, End: {obj_end}")
    # print(f"POS Tags: {pos_tags}")
    # print(f"Dependency Relations: {deprel_tags}")



In [26]:
#makes the boxes wider
wide_layout = widgets.Layout(width="50%")
full_width_layout = widgets.Layout(width="90%")

#create widget boxes
sentence_box = widgets.Textarea(
    placeholder="Enter your sentence here...",
    description="Sentence:",
    layout=full_width_layout
)

subject_box = widgets.Text(
    placeholder="Enter subject entity...",
    description="Subject:",
    layout=wide_layout
)

subject_type_box = widgets.Text(
    placeholder="Enter subject type (e.g., PERSON, ORGANIZATION)...",
    description="Subj Type:",
    layout=wide_layout
)

object_box = widgets.Text(
    placeholder="Enter object entity...",
    description="Object:",
    layout=wide_layout
)

object_type_box = widgets.Text(
    placeholder="Enter object type (e.g., PERSON, ORGANIZATION)...",
    description="Obj Type:",
    layout=wide_layout
)

pos_box = widgets.Textarea(
    placeholder="Enter POS tags (comma-separated)...",
    description="POS:",
    layout=full_width_layout
)

deprel_box = widgets.Textarea(
    placeholder="Enter dependency relations (comma-separated)...",
    description="DepRel:",
    layout=full_width_layout
)

submit_button = widgets.Button(
    description="Run Model",
    button_style="danger"
)


display(sentence_box, subject_box, subject_type_box, object_box, object_type_box, pos_box, deprel_box, submit_button)

submit_button.on_click(on_submit)

Textarea(value='', description='Sentence:', layout=Layout(width='90%'), placeholder='Enter your sentence here.…

Text(value='', description='Subject:', layout=Layout(width='50%'), placeholder='Enter subject entity...')

Text(value='', description='Subj Type:', layout=Layout(width='50%'), placeholder='Enter subject type (e.g., PE…

Text(value='', description='Object:', layout=Layout(width='50%'), placeholder='Enter object entity...')

Text(value='', description='Obj Type:', layout=Layout(width='50%'), placeholder='Enter object type (e.g., PERS…

Textarea(value='', description='POS:', layout=Layout(width='90%'), placeholder='Enter POS tags (comma-separate…

Textarea(value='', description='DepRel:', layout=Layout(width='90%'), placeholder='Enter dependency relations …

Button(button_style='danger', description='Run Model', style=ButtonStyle())

Thinking....


Output(layout=Layout(border='1px solid black', width='90%'))

`Elon Musk is the CEO of Tesla`  

- **Subject:** `Elon Musk`  
- **Subject Type:** `PERSON`  
- **Object:** `Tesla`  
- **Object Type:** `ORGANIZATION`  
- **POS Tags:** `NNP, NNP, VBZ, DT, NN, IN, NNP`  
- **Dependency Relations:** `nsubj, compound, ROOT, det, attr, case, nmod`  